<a href="https://colab.research.google.com/github/NABI-SNU/book/blob/main/tutorials/Session_1_Models/Tutorial1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/NABI-SNU/book/main/tutorials/Session_1_Models/Tutorial1.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# Tutorial 1: Multiple Linear Regression and Polynomial Regression

**Session 1: Models**

**Objective:** Fit linear and polynomial regression models with design matrices.


## Tutorial Objectives

Linear regression predicts a response as a weighted sum of input features:

$$
\mathbf{\hat y} = \mathbf{X}\boldsymbol{\hat\theta}.
$$

In this tutorial, we will use design matrices to fit multiple linear regression and polynomial regression models.

By the end, you will be able to:

- Build a design matrix for regression.
- Estimate regression parameters with ordinary least squares.
- Visualize a fitted plane for two-feature regression.
- Fit polynomial regression models of different orders.
- Compare polynomial fits using mean squared error (MSE).


In [ ]:
# Imports and shared settings
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def evaluate_fits(order_list, mse_list):
  """ Compare the quality of multiple polynomial fits
  by plotting their MSE values.

  Args:
    order_list (list): list of the order of polynomials to be compared
    mse_list (list): list of the MSE values for the corresponding polynomial fit
  """
  fig, ax = plt.subplots()
  ax.bar(order_list, mse_list)
  ax.set(title='Comparing Polynomial Fits', xlabel='Polynomial order', ylabel='MSE')
  plt.show()

def make_design_matrix(x, order):
  """Create the design matrix of inputs for use in polynomial regression

  Args:
    x (ndarray): input vector of shape (n_samples)
    order (scalar): polynomial regression order

  Returns:
    ndarray: design matrix for polynomial regression of shape (samples, order+1)
  """

  # Broadcast to shape (n x 1) so dimensions work
  if x.ndim == 1:
    x = x[:, None]

  #if x has more than one feature, we don't want multiple columns of ones so we assign
  # x^0 here
  design_matrix = np.ones((x.shape[0],1))

  # Loop through rest of degrees and stack columns
  for degree in range(1, order+1):
      design_matrix = np.hstack((design_matrix, x**degree))

  return design_matrix


def plot_fitted_polynomials(x, y, theta_hat):
  """Plot polynomials of different orders.

  Args:
    x (ndarray): input vector of shape (n_samples)
    y (ndarray): vector of measurements of shape (n_samples)
    theta_hat (dict): polynomial regression weights for different orders
  """

  x = np.asarray(x)
  x_grid = np.linspace(x.min() - .5, x.max() + .5)
  orders = sorted(theta_hat.keys())

  plt.figure()

  for order in orders:
    X_design = make_design_matrix(x_grid, order)
    plt.plot(x_grid, X_design @ theta_hat[order])

  plt.ylabel('y')
  plt.xlabel('x')
  plt.plot(x, y, 'C0.')
  plt.legend([f'order {o}' for o in orders], loc=1)
  plt.title('polynomial fits')
  plt.show()


## Multiple Linear Regression

We start with the univariate linear model:

\begin{equation}
y = \theta x + \epsilon
\end{equation}

Here, $\theta$ is the regression parameter and $\epsilon$ is noise. A common way to estimate $\theta$ is to choose the value that minimizes mean squared error (MSE) between model predictions and observed data.

For multiple features, we add one parameter per feature:

\begin{equation}
y = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + ... +\theta_d x_d + \epsilon.
\end{equation}

In matrix form, this becomes:

\begin{equation}
\mathbf{y} = \mathbf{X}\boldsymbol{\theta} + \boldsymbol{\epsilon},
\end{equation}

where $\mathbf{X}$ is the **design matrix**: rows are samples, columns are features, and the first column is often all ones for the intercept.

OLS chooses the parameter vector that minimizes the squared prediction error:

\begin{equation}
\mathcal{L}(\boldsymbol{\theta}) = \|\mathbf{y} - \mathbf{X}\boldsymbol{\theta}\|^2.
\end{equation}

Expanding this expression gives:

\begin{equation}
\mathcal{L}(\boldsymbol{\theta}) = \mathbf{y}^\top\mathbf{y} - 2\boldsymbol{\theta}^\top\mathbf{X}^\top\mathbf{y} + \boldsymbol{\theta}^\top\mathbf{X}^\top\mathbf{X}\boldsymbol{\theta}.
\end{equation}

Taking the derivative with respect to $\boldsymbol{\theta}$ and setting it to zero gives the **normal equations**:

\begin{equation}
\mathbf{X}^\top\mathbf{X}\boldsymbol{\hat\theta} = \mathbf{X}^\top\mathbf{y}.
\end{equation}

If $\mathbf{X}^\top\mathbf{X}$ is invertible, we can solve for the ordinary least squares (OLS) estimate:

\begin{equation}
\boldsymbol{\hat\theta} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}.
\end{equation}

Geometrically, this solution makes the residual vector $\mathbf{y} - \mathbf{X}\boldsymbol{\hat\theta}$ orthogonal to every column of $\mathbf{X}$. In code, we will use a pseudoinverse-based implementation for better numerical robustness.


For this tutorial we will focus on the two-dimensional case ($d=2$), which allows us to easily visualize our results. 

In [ ]:
# Set random seed for reproducibility
np.random.seed(1234)

# Set parameters
theta = [0, -2, -3]
n_samples = 40

# Draw x and calculate y
n_regressors = len(theta)
x0 = np.ones((n_samples, 1))
x1 = np.random.uniform(-2, 2, (n_samples, 1))
x2 = np.random.uniform(-2, 2, (n_samples, 1))
X = np.hstack((x0, x1, x2))
noise = np.random.randn(n_samples)
y = X @ theta + noise


ax = plt.subplot(projection='3d')
ax.plot(X[:,1], X[:,2], y, '.')

ax.set(
    xlabel='$x_1$',
    ylabel='$x_2$',
    zlabel='y'
)
plt.tight_layout()


## Exercise: Ordinary Least Squares Estimator

In this exercise you will implement the OLS approach to estimating $\boldsymbol{\hat\theta}$ from the design matrix $\mathbf{X}$ and measurement vector $\mathbf{y}$. The closed-form solution is $(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$ when $\mathbf{X}^\top\mathbf{X}$ is invertible. In code, `np.linalg.pinv(X) @ y` is a more robust way to compute the same least-squares estimate because it also handles nearly singular design matrices.


In [ ]:
def ordinary_least_squares(X, y):
  """Ordinary least squares estimator for linear regression.

  Args:
    X (ndarray): design matrix of shape (n_samples, n_regressors)
    y (ndarray): vector of measurements of shape (n_samples)

  Returns:
    ndarray: estimated parameter values of shape (n_regressors)
  """
  ######################################################################
  ## TODO for students: solve for the optimal parameter vector using OLS
  # Fill out function and remove
  raise NotImplementedError("Student exercise: solve for theta_hat vector using OLS")
  ######################################################################

  # Compute theta_hat using a numerically robust least-squares solution
  theta_hat = ...

  return theta_hat


theta_hat = ordinary_least_squares(X, y)
print(theta_hat)


After filling in this function, you should see that $\boldsymbol{\hat\theta} = $
```[ 0.13861386, -2.09395731, -3.16370742]```.

In [ ]:
# to_remove solution

def ordinary_least_squares(X, y):
  """Ordinary least squares estimator for linear regression.

  Args:
    X (ndarray): design matrix of shape (n_samples, n_regressors)
    y (ndarray): vector of measurements of shape (n_samples)

  Returns:
    ndarray: estimated parameter values of shape (n_regressors)
  """

  # Compute theta_hat using a numerically robust least-squares solution
  theta_hat = np.linalg.pinv(X) @ y

  return theta_hat


theta_hat = ordinary_least_squares(X, y)
print(theta_hat)


Now that we have our $\boldsymbol{\hat\theta}$, we can obtain $\hat{\mathbf{y}}$ and thus our mean squared error.

In [ ]:
# Compute predicted data
theta_hat = ordinary_least_squares(X, y)
y_hat = X @ theta_hat

# Compute MSE
print(f"MSE = {np.mean((y - y_hat)**2):.2f}")


Finally, the following code will plot a geometric visualization of the data points (blue) and fitted plane.

In [ ]:
theta_hat = ordinary_least_squares(X, y)
y_hat = X @ theta_hat
xx, yy = np.mgrid[-2:2:50j, -2:2:50j]
X_grid = np.column_stack([np.ones(xx.size), xx.ravel(), yy.ravel()])
y_hat_grid = (X_grid @ theta_hat).reshape(xx.shape)

ax = plt.subplot(projection='3d')
ax.plot(X[:, 1], X[:, 2], y, '.')
ax.plot_surface(xx, yy, y_hat_grid, linewidth=0, alpha=0.5, color='C1',
                cmap=plt.get_cmap('coolwarm'))

for i in range(len(X)):
  ax.plot((X[i, 1], X[i, 1]),
          (X[i, 2], X[i, 2]),
          (y[i], y_hat[i]),
          'g-', alpha=.5)

ax.set(
    xlabel='$x_1$',
    ylabel='$x_2$',
    zlabel='y'
)
plt.tight_layout()
plt.show()


## Polynomial Regression

Linear regression predicts outputs as a weighted sum of the inputs:

\begin{equation}
y = \theta_0 + \theta_1 x + \epsilon.
\end{equation}

Polynomial regression keeps the same least-squares fitting machinery, but changes the features in the design matrix. For a third-order polynomial, the model is:

\begin{equation}
y = \theta_0 + \theta_1 x + \theta_2 x^2 + \theta_3 x^3 + \epsilon.
\end{equation}

The order of a polynomial is the highest power of $x$ included in the model. Increasing the order lets the model represent more curved relationships, but it can also make the model more flexible than the data justify.


First, we will simulate some data to practice fitting polynomial regression models. We will generate random inputs $x$ and then compute $y$ according to $y = x^2 - x - 2$, with extra output noise to make the model-fitting exercise closer to a real-life situation. Later, we will also jitter the observed input values to make the sampled points less regular; standard OLS still models the conditional mean of $y$ given the observed $x$.

In [ ]:
# setting a fixed seed to our random number generator ensures we will always
# get the same pseudorandom number sequence
np.random.seed(121)
n_samples = 30
x = np.random.uniform(-2, 2.5, n_samples)  # inputs uniformly sampled from [-2, 2.5)
y = x**2 - x - 2   # computing the outputs

output_noise = 1/8 * np.random.randn(n_samples)
y += output_noise  # adding some output noise

input_noise = 1/2 * np.random.randn(n_samples)
x += input_noise  # adding some input noise

fig, ax = plt.subplots()
ax.scatter(x, y)  # produces a scatter plot
ax.set(xlabel='x', ylabel='y');


## Design Matrix for Polynomial Regression


Now we have the basic idea of polynomial regression and some noisy data, let's begin! The key difference between fitting a linear regression model and a polynomial regression model lies in how we structure the input variables.

Let's go back to one feature for each data point. For linear regression, we used $\mathbf{X} = \mathbf{x}$ as the input data, where $\mathbf{x}$ is a vector where each element is the input for a single data point. To add a constant bias (a y-intercept in a 2-D plot), we use $\mathbf{X} = \big[ \boldsymbol 1, \mathbf{x} \big]$, where $\boldsymbol 1$ is a column of ones.  When fitting, we learn a weight for each column of this matrix. So we learn a weight that multiples with column 1 - in this case that column is all ones so we gain the bias parameter ($+ \theta_0$).

This matrix $\mathbf{X}$ that we use for our inputs is known as a **design matrix**. We want to create our design matrix so we learn weights for $\mathbf{x}^2, \mathbf{x}^3,$ etc. Thus, we want to build our design matrix $X$ for polynomial regression of order $k$ as:

\begin{equation}
\mathbf{X} = \big[ \boldsymbol 1 , \mathbf{x}^1, \mathbf{x}^2 , \ldots , \mathbf{x}^k \big],
\end{equation}

where $\boldsymbol{1}$ is the vector the same length as $\mathbf{x}$ consisting of of all ones, and $\mathbf{x}^p$ is the vector $\mathbf{x}$ with all elements raised to the power $p$. Note that $\boldsymbol{1} = \mathbf{x}^0$ and $\mathbf{x}^1 = \mathbf{x}$.

If we have inputs with more than one feature, we can use a similar design matrix but include all features raised to each power. Imagine that we have two features per data point: $\mathbf{x}_m$ is a vector of one feature per data point and  $\mathbf{x}_n$ is another.  Our design matrix for a polynomial regression would be:

\begin{equation}
\mathbf{X} = \big[ \boldsymbol 1 , \mathbf{x}_m^1, \mathbf{x}_n^1, \mathbf{x}_m^2 , \mathbf{x}_n^2\ldots , \mathbf{x}_m^k , \mathbf{x}_n^k \big],
\end{equation}

## Exercise: Structure Design Matrix

Create a function (`make_design_matrix`) that structures the design matrix given the input data and the order of the polynomial you wish to fit. We will print part of this design matrix for our data and order 5.

In [ ]:
def make_design_matrix(x, order):
  """Create the design matrix of inputs for polynomial regression.

  Args:
    x (ndarray): input vector of shape (n_samples)
    order (int): polynomial regression order

  Returns:
    ndarray: design matrix for polynomial regression of shape (n_samples, order + 1)
  """
  ########################################################################
  ## TODO for students: create the design matrix ##
  # Fill out function and remove
  raise NotImplementedError("Student exercise: create the design matrix")
  ########################################################################

  x = np.asarray(x).reshape(-1, 1)
  design_matrix = np.ones((x.shape[0], 1))

  # Loop through rest of degrees and stack columns (hint: np.hstack)
  for degree in range(1, order + 1):
      design_matrix = ...

  return design_matrix


order = 5
X_design = make_design_matrix(x, order)

print(X_design[0:2, 0:2])


You should see that the printed section of this design matrix is

```
[[ 1.         -1.51194917]
 [ 1.         -0.35259945]]
```

## Fitting Polynomial Regression Models


Now that we have the inputs structured correctly in our design matrix, fitting a polynomial regression is the same as fitting a linear regression model. All of the polynomial structure we need to learn is contained in how the inputs are structured in the design matrix. We can use the OLS solution from Section 1 to estimate the polynomial weights.

## Exercise: Fit Polynomial Regression Models

Here, we will fit polynomial regression models to find the regression coefficients ($\theta_0, \theta_1, \theta_2,$ ...) by solving the least-squares problem. Create a function `solve_poly_reg` that loops over different polynomial orders (up to `max_order`), fits each model, and saves the weights for each order. You may invoke the `ordinary_least_squares` function implemented earlier in this notebook.

We will then qualitatively inspect the quality of our fits for each order by plotting the fitted polynomials on top of the data. To see smooth curves, we evaluate the fitted polynomials on a grid of $x$ values ranging between the largest and smallest inputs in the dataset.

In [ ]:
def solve_poly_reg(x, y, max_order):
  """Fit a polynomial regression model for each order 0 through max_order.

  Args:
    x (ndarray): input vector of shape (n_samples)
    y (ndarray): vector of measurements of shape (n_samples)
    max_order (scalar): max order for polynomial fits

  Returns:
    dict: fitted weights for each polynomial model (dict key is order)
  """

  # Create a dictionary with polynomial order as keys,
  # and np array of theta_hat (weights) as the values
  theta_hats = {}

  # Loop over polynomial orders from 0 through max_order
  for order in range(max_order + 1):

    ##################################################################################
    ## TODO for students: Create design matrix and fit polynomial model for this order
    # Fill out function and remove
    raise NotImplementedError("Student exercise: fit a polynomial model")
    ##################################################################################

    # Create design matrix
    X_design = ...

    # Fit polynomial model
    this_theta = ...

    theta_hats[order] = this_theta

  return theta_hats


max_order = 5
theta_hats = solve_poly_reg(x, y, max_order)

# Visualize
plot_fitted_polynomials(x, y, theta_hats)


## Evaluating Fit Quality


As with linear regression, we can compute mean squared error (MSE) to get a sense of how well the model fits the data.

We compute MSE as:

\begin{equation}
\mathrm{MSE} = \frac 1 N ||\mathbf{y} - \hat{\mathbf{y}}||^2 = \frac 1 N \sum_{i=1}^N (y_i - \hat y_i)^2
\end{equation}

where the predicted values for each model are given by $ \hat{\mathbf{y}} = \mathbf{X}\boldsymbol{\hat\theta}$.

*Which model (i.e. which polynomial order) do you think will have the best MSE?*

## Key Takeaways


* Linear regression generalizes naturally to multiple dimensions
* Linear algebra affords us the mathematical tools to reason and solve such problems beyond the two dimensional case

* To change from a linear regression model to a polynomial regression model, we only have to change how the input data is structured

* We can choose the complexity of the model by changing the order of the polynomial model fit

* Higher order polynomial models tend to have lower MSE on the data they're fit with

**Note**: In practice, multidimensional least squares problems can be solved very efficiently (thanks to numerical routines such as LAPACK).



## Notation

\begin{align}
x &\quad \text{input, independent variable}\\
y &\quad \text{response measurement, dependent variable}\\
\epsilon &\quad \text{measurement error, noise contribution}\\
\theta &\quad \text{regression parameter in a one-feature model}\\
\hat{\theta} &\quad \text{estimated regression parameter in a one-feature model}\\
\mathbf{x} &\quad \text{vector of inputs where each element is a different data point}\\
\mathbf{X} &\quad \text{design matrix}\\
\mathbf{y} &\quad \text{vector of measurements}\\
\mathbf{\hat y} &\quad \text{vector of estimated measurements}\\
\boldsymbol{\theta} &\quad \text{vector of regression parameters}\\
\boldsymbol{\hat\theta} &\quad \text{vector of estimated regression parameters}\\
d &\quad \text{dimensionality of input}\\
N &\quad \text{number of samples}\\
\end{align}

## Suggested Readings

[Introduction to Applied Linear Algebra – Vectors, Matrices, and Least Squares](http://vmls-book.stanford.edu/) by Stephen Boyd and Lieven Vandenberghe